### PySpark: Handling Missing Values
- Dropping Columns
- Dropping Rows
- Multiple Parameters for Dropping Values
- Handling Missing Values Using Mean, Median, Mode

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Missing Values Practice").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/30 17:59:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_spark = spark.read.csv('spark_missing_values.csv', header=True, inferSchema=True)

df_spark.show()

+-------+----+----------+------+
|   Name| Age|Experience|Salary|
+-------+----+----------+------+
|   Gary|  22|         4| 15000|
| Andrew|  21|         8| 65000|
|   Ryan|  22|        14| 80000|
|  Kevin|  20|         5| 20000|
|    Huy|  24|         6| 25000|
|   Alex|  27|         7| 18000|
|Yu-Peng|NULL|      NULL| 40000|
|   NULL|  34|        10| 38000|
|   NULL|  36|      NULL|  NULL|
+-------+----+----------+------+



#### Dropping Columns

In [3]:
# method is very simple: df_sample.drop('column_name')
df_spark.drop('Name').show()

+----+----------+------+
| Age|Experience|Salary|
+----+----------+------+
|  22|         4| 15000|
|  21|         8| 65000|
|  22|        14| 80000|
|  20|         5| 20000|
|  24|         6| 25000|
|  27|         7| 18000|
|NULL|      NULL| 40000|
|  34|        10| 38000|
|  36|      NULL|  NULL|
+----+----------+------+



#### Dropping Rows
- usually used for dropping NULL/NAN values
- like for example, dropping the last two rows
    - since the second last row doesn't have a name 
    - the last row is missing both the name, and the experience and salary as well

In [4]:
# Dropping for na values: .na
    # has drop, fill, replace
    
df_spark.na.drop().show()
# every row that has a single null value will get removed

+------+---+----------+------+
|  Name|Age|Experience|Salary|
+------+---+----------+------+
|  Gary| 22|         4| 15000|
|Andrew| 21|         8| 65000|
|  Ryan| 22|        14| 80000|
| Kevin| 20|         5| 20000|
|   Huy| 24|         6| 25000|
|  Alex| 27|         7| 18000|
+------+---+----------+------+



In [5]:
# drop constraints: how = any or all
    # any: drops a row if any col is na
    # all: drops a row if all the cols are na

df_spark.na.drop(how="all").show()
# as can be seen below, 

+-------+----+----------+------+
|   Name| Age|Experience|Salary|
+-------+----+----------+------+
|   Gary|  22|         4| 15000|
| Andrew|  21|         8| 65000|
|   Ryan|  22|        14| 80000|
|  Kevin|  20|         5| 20000|
|    Huy|  24|         6| 25000|
|   Alex|  27|         7| 18000|
|Yu-Peng|NULL|      NULL| 40000|
|   NULL|  34|        10| 38000|
|   NULL|  36|      NULL|  NULL|
+-------+----+----------+------+



In [6]:
df_spark.na.drop(how="any").show()

+------+---+----------+------+
|  Name|Age|Experience|Salary|
+------+---+----------+------+
|  Gary| 22|         4| 15000|
|Andrew| 21|         8| 65000|
|  Ryan| 22|        14| 80000|
| Kevin| 20|         5| 20000|
|   Huy| 24|         6| 25000|
|  Alex| 27|         7| 18000|
+------+---+----------+------+



#### Thresholds
- is the number of values required in a row, otherwise the row has to be removed
- checks each row if there are enough values to meet the threshold, otherwise the row gets removed

In [7]:
df_spark.na.drop(how="any", thresh=2).show() # checks each row if there is at least 2 values are there, if not the row gets removed

# since in our example, only the last row has 1 value, so only the last row is removed
# the second last row has 2 values, and thus meets the threshold

+-------+----+----------+------+
|   Name| Age|Experience|Salary|
+-------+----+----------+------+
|   Gary|  22|         4| 15000|
| Andrew|  21|         8| 65000|
|   Ryan|  22|        14| 80000|
|  Kevin|  20|         5| 20000|
|    Huy|  24|         6| 25000|
|   Alex|  27|         7| 18000|
|Yu-Peng|NULL|      NULL| 40000|
|   NULL|  34|        10| 38000|
+-------+----+----------+------+



#### Subset
- only dropping values if there are nulls in specific columns
    - Ex: only dropping values if there is a name missing in the row
- useful for important or unique features that cannot be simply imputed like customer id, product ids

In [8]:
df_spark.na.drop(subset=["Experience"]).show() # since there is only one column in the subset parameter, it doesn't do anything

# when using subset, and how parameters: how works differently
# when all is selected, the row gets removed if all of the parameters in subset are missing/Null
# when any is selected, the row gets removed if any of the parameters are missing/null

+------+---+----------+------+
|  Name|Age|Experience|Salary|
+------+---+----------+------+
|  Gary| 22|         4| 15000|
|Andrew| 21|         8| 65000|
|  Ryan| 22|        14| 80000|
| Kevin| 20|         5| 20000|
|   Huy| 24|         6| 25000|
|  Alex| 27|         7| 18000|
|  NULL| 34|        10| 38000|
+------+---+----------+------+



#### Filling Missing Values (Imputation)
- in PySpark the function is to use .fill()
    - `df_example.na.fill('Value_to_be_Filled', ["column_filled_1", "column_filled_2", "column_filled_3", ..., "column_filled_N"])`

In [9]:
df_spark.na.fill(0, ['Experience', 'Age']).show()

# as can be seen below, missing values for experience and age have now been filled in with the replacement value of 0
# while salary is still null, since it was not one of the columns specified to be filled in with missing values in the code above

+-------+---+----------+------+
|   Name|Age|Experience|Salary|
+-------+---+----------+------+
|   Gary| 22|         4| 15000|
| Andrew| 21|         8| 65000|
|   Ryan| 22|        14| 80000|
|  Kevin| 20|         5| 20000|
|    Huy| 24|         6| 25000|
|   Alex| 27|         7| 18000|
|Yu-Peng|  0|         0| 40000|
|   NULL| 34|        10| 38000|
|   NULL| 36|         0|  NULL|
+-------+---+----------+------+



Filling Missing Values Using Mean/Medium/Mode

In [10]:
df_spark.show()

# from the visual below, age, experience and salary are the values that can be imputed using the mean

+-------+----+----------+------+
|   Name| Age|Experience|Salary|
+-------+----+----------+------+
|   Gary|  22|         4| 15000|
| Andrew|  21|         8| 65000|
|   Ryan|  22|        14| 80000|
|  Kevin|  20|         5| 20000|
|    Huy|  24|         6| 25000|
|   Alex|  27|         7| 18000|
|Yu-Peng|NULL|      NULL| 40000|
|   NULL|  34|        10| 38000|
|   NULL|  36|      NULL|  NULL|
+-------+----+----------+------+



In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols = ['Age', 'Experience', 'Salary'], # columns that need to be imputed
    outputCols = ["{}_imputed".format(cols) for cols in ['Age', 'Experience', 'Salary']] 
    # this formatting means that each output cols is the column of the input plus the "_imputed" part at the end, where it gets added on by the .format and {}
    # where cols is extracted from the input columns, this can also be set in a list prior and outside that can be changed by an interface
).setStrategy("mean")
# strategy can differentiate between mean, medium and mode depending on which is the best option

In [13]:
imputer.fit(df_spark).transform(df_spark).show()

# the null values have been replaced by the imputed columns and the mean values of each column

+-------+----+----------+------+-----------+------------------+--------------+
|   Name| Age|Experience|Salary|Age_imputed|Experience_imputed|Salary_imputed|
+-------+----+----------+------+-----------+------------------+--------------+
|   Gary|  22|         4| 15000|         22|                 4|         15000|
| Andrew|  21|         8| 65000|         21|                 8|         65000|
|   Ryan|  22|        14| 80000|         22|                14|         80000|
|  Kevin|  20|         5| 20000|         20|                 5|         20000|
|    Huy|  24|         6| 25000|         24|                 6|         25000|
|   Alex|  27|         7| 18000|         27|                 7|         18000|
|Yu-Peng|NULL|      NULL| 40000|         25|                 7|         40000|
|   NULL|  34|        10| 38000|         34|                10|         38000|
|   NULL|  36|      NULL|  NULL|         36|                 7|         37625|
+-------+----+----------+------+-----------+--------